# 05 — Interprétabilité globale
Comment le modèle XGBoost fonctionne de manière générale ?

1. **Permutation importance** : quelles features sont vraiment utiles au modèle ?
2. **PDP / ICE** : comment les 2 indicateurs les plus discriminants influencent la prédiction

In [ ]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import shap
import sys
sys.path.insert(0, '../src')
from data import load_dataset_split, FEATURES
from sklearn.inspection import permutation_importance

In [ ]:
X_train, X_test, y_train, y_test = load_dataset_split()
model = joblib.load('../models/xgboost.joblib')
print('Jeu de test :', X_test.shape)

## 1. Permutation Importance
On mélange aléatoirement les valeurs d'une feature et on mesure la chute du F1.
Si le F1 chute beaucoup → la feature est importante. Si ça ne change rien → la feature est inutile.

In [ ]:
result = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42, scoring='f1', n_jobs=-1)
perm_df = pd.DataFrame({'feature': FEATURES, 'mean': result.importances_mean, 'std': result.importances_std})
perm_df = perm_df.sort_values('mean', ascending=False)
print('Top 5 features :')
print(perm_df.head(5).to_string(index=False))

In [ ]:
perm_plot = perm_df.sort_values('mean', ascending=True)
colors = ['#EF4444' if v > perm_plot['mean'].median() else '#4C72B0' for v in perm_plot['mean']]
perm_plot.plot.barh(x='feature', y='mean', xerr='std', color=colors, figsize=(9, 7), legend=False,
                    capsize=3, error_kw={'elinewidth': 1.5})
plt.xlabel('Chute du F1 après permutation')
plt.title('Permutation Feature Importance\n(chute de F1 quand la feature est mélangée)', fontweight='bold')
plt.axvline(0, color='black', lw=0.8)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../plots/permutation_importance.png', dpi=150)
plt.show()

## 2. SHAP Summary Plot (Beeswarm)
Chaque point = un compte du jeu de test.
- **Position horizontale** : valeur SHAP (contribution à la prédiction)
- **Couleur** : valeur de la feature (rouge = valeur élevée, bleu = valeur faible)

Permet de voir à la fois l'importance ET la direction : une feature rouge à droite signifie que sa valeur élevée pousse vers bot.

In [ ]:
explainer = shap.TreeExplainer(model.named_steps['model'])
X_test_scaled = model.named_steps['scaler'].transform(X_test)
shap_values = explainer.shap_values(X_test_scaled)
print('SHAP values calculées :', shap_values.shape)

In [ ]:
shap.summary_plot(shap_values, X_test_scaled, feature_names=FEATURES, show=False)
plt.title('SHAP Beeswarm — interprétabilité globale XGBoost', fontweight='bold')
plt.tight_layout()
plt.savefig('../plots/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. PDP / ICE — 2 indicateurs clés
`retweet_followers_ratio` et `reply_status_ratio` sont les 2 indicateurs les plus importants selon la permutation importance.

- `retweet_followers_ratio` : 83% des bots ont une valeur à 0 (ils ne retweetent pas ou ont très peu de followers). Les humains ont une médiane à 1.78. Une valeur proche de 0 est donc un signal bot fort.
- `reply_status_ratio` : les bots répondent massivement de façon automatisée. Médiane bots = 0.60 vs humains = 0.11 — la proportion de réponses est 5× plus élevée chez les bots.

In [ ]:
top2 = ['retweet_followers_ratio', 'reply_status_ratio']
print('Features analysées :', top2)

In [ ]:
feat_configs = [
    ('retweet_followers_ratio', 0.0, X_test['retweet_followers_ratio'].quantile(0.97)),
    ('reply_status_ratio',      0.0, min(X_test['reply_status_ratio'].quantile(0.97), 2.5)),
]

idx_bots   = y_test[y_test==1].sample(150, random_state=42).index
idx_humans = y_test[y_test==0].sample(150, random_state=42).index
X_s = X_test.loc[idx_bots.append(idx_humans)].copy()
y_s = y_test.loc[idx_bots.append(idx_humans)].values

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (feat, lo, hi) in zip(axes, feat_configs):
    grid = np.linspace(lo, hi, 80)
    ice = np.zeros((len(X_s), len(grid)))
    for j, val in enumerate(grid):
        X_tmp = X_s.copy(); X_tmp[feat] = val
        ice[:, j] = model.predict_proba(X_tmp)[:, 1]
    for i in np.where(y_s==0)[0]:
        ax.plot(grid, ice[i], color='#06B6D4', alpha=0.12, lw=0.7)
    for i in np.where(y_s==1)[0]:
        ax.plot(grid, ice[i], color='#EF4444', alpha=0.12, lw=0.7)
    ax.plot(grid, ice.mean(axis=0),         color='#F59E0B', lw=3.5, label='PDP global')
    ax.plot(grid, ice[y_s==1].mean(axis=0), color='#EF4444', lw=2.2, ls='--', label='PDP bots')
    ax.plot(grid, ice[y_s==0].mean(axis=0), color='#06B6D4', lw=2.2, ls='--', label='PDP humains')
    x_real  = X_s[feat].values
    y_proba = model.predict_proba(X_s)[:, 1]
    ax.scatter(x_real[y_s==0], y_proba[y_s==0], color='#06B6D4', alpha=0.5, s=22, edgecolors='white', lw=0.3, zorder=7)
    ax.scatter(x_real[y_s==1], y_proba[y_s==1], color='#EF4444', alpha=0.5, s=22, edgecolors='white', lw=0.3, zorder=7)
    ax.set_title(feat, fontweight='bold')
    ax.set_xlabel(feat); ax.set_ylabel('Proba bot')
    ax.set_xlim(lo, hi); ax.set_ylim(-0.05, 1.1)
    ax.grid(alpha=0.3); ax.legend(fontsize=9)

fig.suptitle('PDP / ICE — retweet_followers_ratio & reply_status_ratio', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../plots/pdp_top2.png', dpi=150)
plt.show()

## Conclusion — ce que le modèle a appris
Les deux indicateurs les plus discriminants visuellement sont des **ratios comportementaux** :

- `friends_followers_ratio` : un bot suit massivement des comptes mais personne ne le suit en retour → ratio élevé (médiane bots = 6.1 vs humains = 1.0). Au-delà d'un seuil ~3–5, la probabilité bot monte fortement.
- `reply_status_ratio` : les bots répondent de façon automatisée et répétitive → proportion de réponses très élevée (médiane bots = 0.60 vs humains = 0.11). La séparation est nette au-delà de ~0.2.

Les courbes PDP/ICE confirment une **rupture claire** : en dessous des seuils, bots et humains se confondent ; au-delà, les bots se distinguent de façon très marquée.